# Notebook Introduction
This notebook covers my implementation for classifying twitter tweets as disaster/non-disaster. I've developed a simple distilBERT classification model which achieved a public score of 0.82 on the Kaggle Leaderboard.

Other things covered in this notebook is training on GPUs, and parallelizing training on GPUs.

Please do leave comments if some part of the code is unclear - would like to improve these notebooks and hopefully have someone else learn through my work :)

# Imports

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import cross_val_score
import torch
import transformers as ppb
import warnings
import re

# Loading the Data

In [ ]:
df_train = pd.read_csv("/kaggle/input/nlp-getting-started/train.csv")

# Load BERT

Loading the BERT model. I've used the DistilBert model in this case to reduce computation time.

In [ ]:
from transformers import DistilBertForSequenceClassification, DistilBertTokenizer

# Load pretrained tokenizer
tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')

# Load pretrained model with classification head
model = DistilBertForSequenceClassification.from_pretrained(
    'distilbert-base-uncased',
    num_labels=2  # binary classification: disaster / non-disaster
)



# Pre-processing Text

I've highlighted some of exploratory analysis for this dataset. I noticed a lot of tweets contained mentions, URLs and hashtags. Since BERT would interpret @BBCNews or other accounts as separate words, and likely would not have a lot of these available in its vocabulary, I generalised mentions and URLs to tags \<URL\>, \<MENTION\>. While the URL/Mention itself may not provide a lot of useful information to the model, the presence/absence of it could help the model to classify between disaster/non-disaster.
Lastly, I removed punctuations from the tweets as I did not think it provided any useful information to the model.

For the future, I believe we could do some study on the length of the tweets, number of punctuations (maybe exclamation marks could indicate adversity) and create features to feed into the model for the same.

## Check tokenized text created by BERT

In [ ]:
df_train['tokenized'] = df_train['text'].apply((lambda x: tokenizer.tokenize(x)))

In [ ]:
df_train.head()

In [ ]:
(df_train.head(10)  # adjust number of rows
   .style.set_properties(subset=['text'],  # replace 'text' with your tweet column name
                         **{'white-space': 'pre-wrap', 'word-wrap': 'break-word'}))

## Check URLs in text

In [ ]:
def check_url(text):
    url_pattern = re.compile(r'http[s]?://\S+|www\.\S+')
    return bool(url_pattern.search(text))

In [ ]:
df_train['contain_url'] = df_train['text'].apply(lambda x: check_url(x))

In [ ]:
# df_train[df_train['contain_url'] == 1]

(df_train[df_train['contain_url'] == 1].head(10)  # adjust number of rows
   .style.set_properties(subset=['text'],  # replace 'text' with your tweet column name
                         **{'white-space': 'pre-wrap', 'word-wrap': 'break-word'}))

In [ ]:
df_train[df_train['contain_url'] == 1]['target'].value_counts()

In [ ]:
# Here We also Use Regular Expressions to Remove URLs from Text or Whole Corpus.
def remove_url(text):
    pattern = re.compile(r'https?://\S+|www\.\S+')
    return pattern.sub(r'[URL]', text)

In [ ]:
df_train['cleaned_text'] = df_train['text'].apply(lambda x: remove_url(x))

In [ ]:
# df_train[df_train['contain_url'] == 1]

(df_train[df_train['contain_url'] == 1].head(10)  # adjust number of rows
   .style.set_properties(subset=['text'],  # replace 'text' with your tweet column name
                         **{'white-space': 'pre-wrap', 'word-wrap': 'break-word'}))

## Check mentions

In [ ]:
def contains_mention(text):
    mention_pattern = re.compile(r'@[A-Za-z_]\w*')
    return bool(mention_pattern.search(text))

In [ ]:
df_train['contain_mentions'] = df_train['text'].apply(lambda x: contains_mention(x))

In [ ]:
df_train[df_train['contain_mentions'] == 1]['target'].value_counts()

In [ ]:
# Here We also Use Regular Expressions to Remove URLs from Text or Whole Corpus.
def remove_mentions(text):
    pattern = re.compile(r'@[A-Za-z_]\w*')
    return pattern.sub(r'[MENTION]', text)

In [ ]:
df_train['cleaned_text'] = df_train['cleaned_text'].apply(lambda x: remove_mentions(x))

In [ ]:
(df_train[df_train['contain_mentions'] == 1].head(10)  # adjust number of rows
   .style.set_properties(subset=['text'],  # replace 'text' with your tweet column name
                         **{'white-space': 'pre-wrap', 'word-wrap': 'break-word'}))

## Clean text to remove non-alphanumeric characters

In [ ]:
import re

def clean_text(text):
    return re.sub(r'[^A-Za-z0-9 ]+', '', text)

In [ ]:
df_train['cleaned_text'] = df_train['cleaned_text'].apply(clean_text)

In [ ]:
df_train['cleaned_tokenized'] = df_train['cleaned_text'].apply((lambda x: tokenizer.tokenize(x)))

In [ ]:
(df_train[df_train['contain_url'] == 1].head(10)  # adjust number of rows
   .style.set_properties(subset=['text'],  # replace 'text' with your tweet column name
                         **{'white-space': 'pre-wrap', 'word-wrap': 'break-word'}))

## Analyse pre-processing
Below I create some visualizations to see if there's any difference between the top tokens that appear in disaster vs non-disaster tweets. The first visualization was including a lot of stop words and common words, so I later analysed the log-odds of the words and could see clear distinctions between top words for disaster tweets vs non-disaster tweets. BERT initializes these words with weights, so common words would show up less anyways and there's no need to handle these words

In [ ]:
import pandas as pd
from collections import Counter
import matplotlib.pyplot as plt
import seaborn as sns

# Separate token lists by class
tokens_disaster = df_train[df_train['target'] == 1]['cleaned_tokenized'].explode()
tokens_nondisaster = df_train[df_train['target'] == 0]['cleaned_tokenized'].explode()

# Count token frequencies
disaster_counts = Counter(tokens_disaster)
nondisaster_counts = Counter(tokens_nondisaster)

# Convert to DataFrames
df_disaster = pd.DataFrame(disaster_counts.items(), columns=['token', 'disaster_count'])
df_nondisaster = pd.DataFrame(nondisaster_counts.items(), columns=['token', 'nondisaster_count'])

# Merge the two for comparison
token_dist = pd.merge(df_disaster, df_nondisaster, on='token', how='outer').fillna(0)

# Optionally, sort by most frequent in either class
token_dist['total'] = token_dist['disaster_count'] + token_dist['nondisaster_count']
token_dist = token_dist.sort_values(by='total', ascending=False)

# Plot top 20 disaster tokens vs non-disaster tokens
top_n = 50

plt.figure(figsize=(14, 6))

# Plot for disaster tweets
plt.subplot(1, 2, 1)
sns.barplot(data=token_dist.nlargest(top_n, 'disaster_count'), x='disaster_count', y='token', color='red')
plt.title('Top Tokens in Disaster Tweets')

# Plot for non-disaster tweets
plt.subplot(1, 2, 2)
sns.barplot(data=token_dist.nlargest(top_n, 'nondisaster_count'), x='nondisaster_count', y='token', color='blue')
plt.title('Top Tokens in Non-Disaster Tweets')

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np

def compute_log_odds(count_dis, count_nondis, alpha=0.01):
    # Vocabulary
    vocab = set(count_dis.keys()) | set(count_nondis.keys())
    
    total_dis = sum(count_dis.values())
    total_nondis = sum(count_nondis.values())
    
    log_odds = {}
    for word in vocab:
        # counts with smoothing
        dis = count_dis.get(word, 0) + alpha
        nondis = count_nondis.get(word, 0) + alpha
        
        # proportions
        p_dis = dis / (total_dis + alpha * len(vocab))
        p_nondis = nondis / (total_nondis + alpha * len(vocab))
        
        # log odds ratio
        log_odds[word] = np.log(p_dis / p_nondis)
    
    return log_odds

log_odds = compute_log_odds(disaster_counts, nondisaster_counts)


In the chart below, we can see a clear distinction between tokens that show up for disaster vs non-disaster

In [ ]:
# Top tokens for disaster
sorted_disaster = sorted(log_odds.items(), key=lambda x: x[1], reverse=True)[:20]

# Top tokens for non-disaster
sorted_nondisaster = sorted(log_odds.items(), key=lambda x: x[1])[:20]

print("🔴 Disaster tokens:")
for w, score in sorted_disaster:
    print(f"{w:15} {score:.2f}")

print("\n🔵 Non-disaster tokens:")
for w, score in sorted_nondisaster:
    print(f"{w:15} {score:.2f}")


# Model

In [ ]:
def preprocess_text(text):
    text = remove_url(text)
    text = remove_mentions(text)
    text = clean_text(text)
    return text

Create dataloader.
Max length = 84 is decided on the basis of the maximum size of the tweets after tokenization

In [ ]:
from torch.utils.data import Dataset, DataLoader
import torch

class TweetDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=84):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])

        # Apply preprocessing here
        text = preprocess_text(text)

        encoding = self.tokenizer.encode_plus(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt'
        )

        item = {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten()
        }

        if self.labels is not None:
            label = self.labels[idx]
            item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)

        return item

# Hyperparater + k-fold

Functions to run hyperparameter tuning along with k-fold cross validation to get the best combination of hyperparameters.
The grid tested on was:
- learning_rates = [1e-5, 2e-5, 3e-5]
- batch_sizes = [16, 32]
- epochs_list = [2, 3, 4]

Results are unavailable in this code run but the best combination was: learning_rates = 1e-5, batch_sizes = 32, epochs_list = 3

The entire experimentation process was run on 2 GPUs - below code parallelizes the processes.

In [ ]:
import torch
from transformers import DistilBertForSequenceClassification
from torch.optim import AdamW
from sklearn.metrics import f1_score
from torch.utils.data import DataLoader
import numpy as np

def train_and_eval(train_loader, val_loader, lr, epochs, device):
    model = DistilBertForSequenceClassification.from_pretrained('distilbert-base-uncased', num_labels=2)
    model.to(device)
    optimizer = AdamW(model.parameters(), lr=lr)

    # Mixed precision support for T4
    scaler = torch.cuda.amp.GradScaler()

    for epoch in range(epochs):
        model.train()
        for batch in train_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            optimizer.zero_grad()
            with torch.cuda.amp.autocast():  # mixed precision
                outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
                loss = outputs.loss
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

    # Validation
    model.eval()
    preds, true_labels = [], []
    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            outputs = model(input_ids, attention_mask=attention_mask)
            pred = torch.argmax(outputs.logits, dim=1)

            preds.extend(pred.cpu().numpy())
            true_labels.extend(labels.cpu().numpy())

    return f1_score(true_labels, preds)


In [ ]:
from sklearn.model_selection import KFold

def kfold_evaluate(texts, labels, tokenizer, lr, batch_size, epochs, k, device):
    kf = KFold(n_splits=k, shuffle=True, random_state=42)
    fold_scores = []

    for fold, (train_idx, val_idx) in enumerate(kf.split(texts)):
        print(f"[GPU {device}] Fold {fold+1}/{k}")

        train_dataset = TweetDataset([texts[i] for i in train_idx],
                                     [labels[i] for i in train_idx],
                                     tokenizer)
        val_dataset = TweetDataset([texts[i] for i in val_idx],
                                   [labels[i] for i in val_idx],
                                   tokenizer)

        train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
        val_loader = DataLoader(val_dataset, batch_size=batch_size)

        f1 = train_and_eval(train_loader, val_loader, lr, epochs, device)
        fold_scores.append(f1)

    return np.mean(fold_scores)


In [ ]:
from itertools import product

# learning_rates = [1e-5, 2e-5, 3e-5]
# batch_sizes = [16, 32]
# epochs_list = [2, 3, 4]
learning_rates = [1e-5]
batch_sizes = [32]
epochs_list = [3]

# All possible combinations
hyperparameter_grid = list(product(learning_rates, batch_sizes, epochs_list))
print("Total experiments:", len(hyperparameter_grid))

In [ ]:
from multiprocessing import Pool
import torch

def run_experiment(params):
    lr, batch_size, epochs, gpu_id = params
    device = torch.device(f"cuda:{gpu_id}")

    print(f"\n[GPU {gpu_id}] Running LR={lr}, Batch={batch_size}, Epochs={epochs}")
    avg_f1 = kfold_evaluate(df_train['text'], df_train['target'], tokenizer, lr, batch_size, epochs, k=5, device=device)

    return {'lr': lr, 'batch_size': batch_size, 'epochs': epochs, 'avg_f1': avg_f1}


In [ ]:
from joblib import Parallel, delayed

# Assign jobs to GPUs round-robin
jobs = [(lr, bs, ep, i % 2) for i, (lr, bs, ep) in enumerate(hyperparameter_grid)]

if __name__ == "__main__":
    results = Parallel(n_jobs=2, backend="loky")(
        delayed(run_experiment)(params) for params in jobs
    )

    # Sort by best F1
    results = sorted(results, key=lambda x: x['avg_f1'], reverse=True)
    print("\n===== Hyperparameter Tuning Results =====")
    for r in results:
        print(r)
    print("\nBest configuration:", results[0])


# Predict and Submit

Run the final model with the best hyperparameter combinations and generate submission.csv

In [ ]:

df_test = pd.read_csv("/kaggle/input/nlp-getting-started/test.csv")
df_train = pd.read_csv("/kaggle/input/nlp-getting-started/train.csv")


from transformers import DistilBertForSequenceClassification, DistilBertTokenizer

# Load pretrained tokenizer
tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')

# Load pretrained model with classification head
model = DistilBertForSequenceClassification.from_pretrained(
    'distilbert-base-uncased',
    num_labels=2  # binary classification: disaster / non-disaster
)



In [ ]:
train_dataset = TweetDataset(df_train['text'].tolist()
                             , df_train['target'].tolist(), tokenizer)
train_dataloader = DataLoader(train_dataset, batch_size=32, shuffle=True)
lr = 1e-5
epochs= 3
test_dataset = TweetDataset(df_test['text'].tolist()
                             , None, tokenizer)
test_dataloader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [ ]:
import torch
from transformers import DistilBertForSequenceClassification
from torch.optim import AdamW
from sklearn.metrics import f1_score
from torch.utils.data import DataLoader
import numpy as np

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)
optimizer = AdamW(model.parameters(), lr=lr)

# Mixed precision support for T4
scaler = torch.cuda.amp.GradScaler()

for epoch in range(epochs):
    model.train()
    for batch in train_dataloader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        optimizer.zero_grad()
        with torch.cuda.amp.autocast():  # mixed precision
            outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
            loss = outputs.loss
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()




In [ ]:
# Generate predictions
model.eval()
preds = []

with torch.no_grad():
    for batch in test_dataloader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)

        outputs = model(input_ids, attention_mask=attention_mask)
        pred = torch.argmax(outputs.logits, dim=1)
        preds.extend(pred.cpu().numpy())

# Convert to pandas DataFrame for submission
submission_df = pd.DataFrame({
    'id': df_test['id'],
    'target': preds
})
submission_df.to_csv("submission.csv", index=False)

# Future Work
Would like to try out additional features in the future like length of tweet, number of emojis, number of mentions, etc.
Would like to try adding additional layers to the model to see if it would improve the results